# Import libraries

In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time


# Fetch Data

In [2]:
# Read data from the CSV for California hospitals
cal_detail = pd.read_csv('../data/raw/California/CA - Final_PRA_29194 - 2026-04-08.xlsx - Detail.csv')
cal_geocode = pd.read_csv('../data/handmade/California/California_Hospital_Geocodes.csv')

# Read data from the CSV files for Colorado Hospitals
col_detail = pd.read_csv('../data/raw/Colorado/CO - Data_Request_-_Michael_Nolan-_03.30.2026_-_Facility_Occurrences_CY_23-25xlsx.xlsx - Data.csv')
col_geocode = pd.read_csv('../data/handmade/Colorado/Colorado_Hospital_Geocodes.csv')

# Read data from the CSV files for Washington Hospitals
wa_detail = pd.read_csv("../data/raw/Washington/WA-downloadable_ae - 2026-04-07.csv")
wa_geocode = pd.read_excel("../data/raw/Washington/Facilities.xlsx")


# Merge California data with their Geocodes

### Merge the data

In [ ]:
cal_detail['ASPEN_FACID'] = cal_detail['ASPEN_FACID'].astype(str).str.strip()
cal_geocode['ASPEN_FACID'] = cal_geocode['ASPEN_FACID'].astype(str).str.strip()

# FIX: deduplicate on ASPEN_FACID to prevent row inflation
cal_geo_dedup = (
    cal_geocode[['ASPEN_FACID', 'LATITUDE', 'LONGITUDE']]
    .drop_duplicates(subset=['ASPEN_FACID'])
    .rename(columns={'LATITUDE': 'LAT', 'LONGITUDE': 'LON'})
)

cal_merged = cal_detail.merge(cal_geo_dedup, on='ASPEN_FACID', how='left')
cal_merged.head()

,INTAKEID,RECVDATE,AE_CATEGORY,AE_SHORT,ADVERSE_EVENT,FINDING_DETAIL,ASPEN_FACID,NAME,ASPEN_FACTYPE_LABEL,LAT,LON
0,CA00819364,4-Jan-23,04 - Care Management Events,17: Stage 3/4 Pressure Ulcer,"17: Stage 3 or 4 Pressure Ulcer, Acquired Afte...",Z2_Missing,CA220000019,ZUCKERBERG SAN FRANCISCO GENERAL HOSP & TRAUMA...,General Acute Care Hospital,37.756161,-122.405665
1,CA00819439,4-Jan-23,04 - Care Management Events,17: Stage 3/4 Pressure Ulcer,"17: Stage 3 or 4 Pressure Ulcer, Acquired Afte...",Z2_Missing,CA030000901,MERCY HOSPITAL OF FOLSOM,General Acute Care Hospital,38.670210,-121.145382
2,CA00819831,9-Jan-23,04 - Care Management Events,17: Stage 3/4 Pressure Ulcer,"17: Stage 3 or 4 Pressure Ulcer, Acquired Afte...",Z2_Missing,CA030000123,MEMORIAL MEDICAL CENTER,General Acute Care Hospital,37.668762,-120.974656
3,CA00820220,9-Jan-23,04 - Care Management Events,17: Stage 3/4 Pressure Ulcer,"17: Stage 3 or 4 Pressure Ulcer, Acquired Afte...",Z2_Missing,CA220000031,UCSF MEDICAL CENTER,General Acute Care Hospital,37.784381,-122.439627
4,CA00820763,11-Jan-23,04 - Care Management Events,17: Stage 3/4 Pressure Ulcer,"17: Stage 3 or 4 Pressure Ulcer, Acquired Afte...",Z2_Missing,CA030000901,MERCY HOSPITAL OF FOLSOM,General Acute Care Hospital,38.670210,-121.145382


### Save CSV

In [ ]:
cal_merged.to_csv("../data/processed/California/CA-Hospital Data with Geocodes.csv", index=False)


# Merge Colorado data with their Geocodes

### Merge the data

In [ ]:
col_detail['Facility ID'] = col_detail['Facility ID'].astype(str).str.strip()
col_geocode['Facility_ID'] = col_geocode['Facility_ID'].astype(str).str.strip()

col_geo_slim = (
    col_geocode[['Facility_ID', 'LAT', 'LON']]
    .rename(columns={'Facility_ID': 'Facility ID'})
)

col_merged = col_detail.merge(col_geo_slim, on='Facility ID', how='left')
col_merged.head()


,Facility ID,Facility Name,Facility Type Code,Facility Type Name,Facility Type Abbreviation,Facility Admin Name,Bed License Total,Owner Company,Facility Operating Status,Facility Address,Facility County,Facility City,Facility Phone,Occurrence ID,Type of Occurrence,Occurrence Date,Occurrence Description,Occurrence Description_Overflow1,LAT,LON
0,10C962,# 1 HOME CARE AGENCY CORPORATION,05D,Home Care Agency-Personal Care/Homemaker (Medi...,HCA-PHS,"Vaynshteyn, Samuil",0,# 1 HOME CARE AGENCY CORPORATION,Active,"10200 E GIRARD AVE BLDG A STE 103, DENVER, CO...",Denver,DENVER,(303) 306-0404,2310C962001,Misappropriation of Property,4/24/2023,"DESCRIPTION OF OCCURRENCE: On 4/24/23, a clie...",NaN,39.656151,-104.868362
1,0506AT,198 E GALATEA,S41,Residential Care Facility for the Developmenta...,RCF-DD,"De Maria, Jacqueline (JR)",8,STATE OF COLORADO,Active,"198 E GALATEA DR, PUEBLO WEST, CO 81007-",Pueblo,PUEBLO WEST,(719) 585-4001,230506AT001,Neglect,3/14/2023,"DESCRIPTION OF OCCURRENCE: On 3/14/23, a staf...",NaN,38.324306,-104.734924
2,0506AT,198 E GALATEA,S41,Residential Care Facility for the Developmenta...,RCF-DD,"De Maria, Jacqueline (JR)",8,STATE OF COLORADO,Active,"198 E GALATEA DR, PUEBLO WEST, CO 81007-",Pueblo,PUEBLO WEST,(719) 585-4001,230506AT002,Physical Abuse,3/21/2023,"DESCRIPTION OF OCCURRENCE: On 3/21/23, staff ...",NaN,38.324306,-104.734924
3,0506AT,198 E GALATEA,S41,Residential Care Facility for the Developmenta...,RCF-DD,"De Maria, Jacqueline (JR)",8,STATE OF COLORADO,Active,"198 E GALATEA DR, PUEBLO WEST, CO 81007-",Pueblo,PUEBLO WEST,(719) 585-4001,230506AT003,Physical Abuse,3/27/2023,DESCRIPTION OF OCCURRENCE: On 3/27/23 during ...,NaN,38.324306,-104.734924
4,0506AT,198 E GALATEA,S41,Residential Care Facility for the Developmenta...,RCF-DD,"De Maria, Jacqueline (JR)",8,STATE OF COLORADO,Active,"198 E GALATEA DR, PUEBLO WEST, CO 81007-",Pueblo,PUEBLO WEST,(719) 585-4001,230506AT004,Physical Abuse,5/8/2023,"DESCRIPTION OF OCCURRENCE: On 5/8/23, staff r...",NaN,38.324306,-104.734924


### Save CSV

In [ ]:
col_merged.to_csv("../data/processed/Colorado/CO-Hospital Data with Geocodes.csv", index=False)


# Merge Washington data with their Geocodes

## Get the locations and use Google Maps API to fetch geocodes and merge them

In [6]:
# KEEP ONLY REQUIRED COLUMNS
wa_geocode = wa_geocode[
    [
        "Credential Number",
        "Facility: Mailing Address"
    ]
]

# Remove duplicate credential numbers
wa_geocode = wa_geocode.drop_duplicates(
    subset="Credential Number"
)

# LEFT MERGE
merged_df = wa_detail.merge(
    wa_geocode,
    left_on="Facility Credential",
    right_on="Credential Number",
    how="left"
)

assert len(wa_detail) == len(merged_df), \
    "ERROR: Row count changed after merge!"

# INITIALIZE GEOPY
geolocator = Nominatim(
    user_agent="hospital_geocoder",
    timeout=10
)


# Nominatim usage policy recommends no more than 1 request/second.
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.5,
    max_retries=3,
    error_wait_seconds=10
)

# GEOCODE FUNCTION
def get_coordinates(address):

    if pd.isna(address):
        return (None, None)

    address = str(address).strip()

    if address == "":
        return (None, None)

    try:
        location = geocode(address)

        if location is None:
            return (None, None)

        return (
            location.latitude,
            location.longitude
        )

    except Exception as e:
        print(f"Error geocoding:\n{address}\n{e}")
        return (None, None)

# GEOCODE ONLY UNIQUE ADDRESSES
unique_addresses = (
    merged_df["Facility: Mailing Address"]
    .dropna()
    .unique()
)

print(f"\nUnique addresses to geocode: {len(unique_addresses)}")

geocode_cache = {}

for i, address in enumerate(unique_addresses, start=1):
    geocode_cache[address] = get_coordinates(address)
    if i % 50 == 0:
        print(f"Processed {i}/{len(unique_addresses)} addresses")

print("Geocoding complete.")

# ADD LATITUDE AND LONGITUDE
merged_df["LAT"] = merged_df["Facility: Mailing Address"].map(
    lambda x: geocode_cache.get(x, (None, None))[0]
)
merged_df["LON"] = merged_df["Facility: Mailing Address"].map(
    lambda x: geocode_cache.get(x, (None, None))[1]
)

# VERIFY ROW COUNT
assert len(wa_detail) == len(merged_df), \
    "ERROR: Row count changed after geocoding!"

print(f"Final row count: {len(merged_df)}")



Unique addresses to geocode: 142
Processed 50/142 addresses
Processed 100/142 addresses
Geocoding complete.
Final row count: 188160


## Save CSV

In [6]:
merged_df.to_csv("../data/processed/Washington/WA-Hospital Data with Geocodes.csv", index=False)

In [12]:
# Get unique facility names where LAT is null
temp_df = merged_df[merged_df['LAT'].isnull()]
null_lat_facilities = temp_df['Facility Name'].unique()
null_lat_ids = temp_df['Facility Credential'].dropna().unique()

print(f"Total number of facilities: {merged_df['Facility Credential'].nunique()} \
\nTotal number of facilities without geocodes: {temp_df['Facility Credential'].nunique()} \
\nTotal number of facilities with geocodes: {int(merged_df['Facility Credential'].nunique())-int(temp_df['Facility Credential'].nunique())}")
print(null_lat_facilities)
print("="*30)
print(null_lat_ids)

Total number of facilities: 489 
Total number of facilities without geocodes: 351 
Total number of facilities with geocodes: 138
['Acute Pain Therapies' 'Advanced Dermatology and Skin Surgery'
 'Advanced Endoscopy Center dba Salmon Creek Surgery Center'
 'Aesthetic and General Dermatology of Seattle'
 'Aesthetic Facial Body Plastic Surgery'
 'Aesthetic Plastic Surgical Center'
 'Aesthetic Surgery Centre and Med Spa' 'Aesthetica Clinique LLC'
 'Ageless' 'Allure Esthetic' 'Allure Laser Center'
 'Allure Laser Center and Medispa' 'Amadi Aesthetics Plastic Surgery'
 'Anderson Sobel Cosmetic Surgery' 'Anesis Bel-Red ASC, LLC'
 'Anesis Spokane ASC' 'Apex Spine ASC' 'Arlington Surgery Center'
 'Artistic Plastic Surgery Center' 'ASC/Endoscopy Center'
 'Athenix Advanced Plastic Surgery and Aesthetic Centers'
 'Auburn Surgery Center' 'Avalon Clinic' 'Aysel K Sanderson MD'
 'Bel Red Ambulatory Surgical Facility' 'Bellevue Plastic Surgery Center'
 'Bellevue Surgery Center' 'Bellingham Ambulatory Su

In [10]:
print(type(null_lat_ids))

<class 'numpy.ndarray'>


In [13]:
asf = 0
cbc = 0
hac = 0
si = 0
no = 0

for id in null_lat_ids:
    if "ASF" in id:
        asf += 1
    elif "CBC" in id:
        cbc += 1
    elif "HAC" in id:
        hac += 1
    elif "SI" in id:
        si += 1
    else:
        no += 1

print(f"ASF = {asf} \nCBC = {cbc} \nHAC = {hac} \nSI = {si} \nNan = {no}")


ASF = 314 
CBC = 21 
HAC = 8 
SI = 8 
Nan = 0
